# Liquid Cu — Δ-learning MLIP pipeline **validation** ("the HCN of metals")

Validate the DeePMD pipeline on a **metallic, periodic, condensed phase**. No reaction, no CV,
no OPES — the observables are **g(r)**, **self-diffusion D**, and force/energy RMSE vs CP2K.

**Architecture (CPU-feasible):**
- **EMT** (ASE built-in) is both the *explorer* (generates liquid configs for free) and the
  *baseline* of a **Δ-model**: DeePMD is trained on Δ = (E_CP2K − E_EMT, F_CP2K − F_EMT).
- **CP2K** single-points label a decorrelated subset (~150–300 frames, resumable, overnight on CPU).
- Production calculator = `SumCalculator([EMT, DP(Δ)])`. Committee σ is unaffected — the
  deterministic baseline cancels in the force deviation.
- **Train small (32 Cu, ~7.5 Å box), run big (2×2×2 → 256 atoms)** — the molten-salt paper's
  strategy (they train on the FPMD cell, produce on a 3×3×3 supercell).

**Paper-derived settings used here**
| choice | source |
|---|---|
| loss prefs `pref_e 0.02→1, pref_f 1000→1`, **no virial** | Xu et al. §2.2, Fig. S3 |
| ~100k steps, convergence ≈50k; RMSE targets <1.3 meV/atom, <45 meV/Å | Xu et al. §3.1.2 |
| rcut near 6–7 Å, `sel` sized to neighbor count + margin | Xu et al. §2.2 (they chose 7.0 Å, sel≈122) |
| train small / replicate for production; NVT equil → production; MSD→D (Einstein) | Xu et al. §2.3, §3.4 |
| committee of 4 seeds; max force deviation ε (Eq. 1); candidate window 0.1–1.0 eV/Å | Achar et al. (RAL) §2.1, Step 7 |
| label sanity filters (SCF converged, force ceiling, ΔE/atom bound) | Achar et al. Step 7, adapted for a smeared metal |

Reference = **CP2K only**. Never mix CP2K- and VASP-labeled frames.


## 1 · Environment
Kernel **is** the `ch4mlip` env. Scripts run via `!{PY}`; data persists under `CU_ROOT`.

In [ ]:
import sys, os, glob, shutil
PY = sys.executable
ENV_PREFIX = os.path.dirname(os.path.dirname(PY))
os.environ['CU_ROOT'] = os.path.expanduser('~/cu-mlip')
os.makedirs(os.environ['CU_ROOT'], exist_ok=True)
print('kernel python :', PY)
print('env prefix    :', ENV_PREFIX)
print('CU_ROOT       :', os.environ['CU_ROOT'])

## 2 · Verify the env
`deepmd` + `ase` are required now; `cp2k` binary is only needed at Stage B (labeling).

In [ ]:
import ase, deepmd, numpy
print('ase   ', ase.__version__)
print('deepmd', deepmd.__version__)
print('numpy ', numpy.__version__)
import shutil
cp2k = next((shutil.which(b) for b in ('cp2k.psmp','cp2k.ssmp','cp2k') if shutil.which(b)), None)
print('cp2k  ', cp2k or 'NOT FOUND — `conda install -c conda-forge cp2k` before Stage B')
print('CP2K_DATA_DIR:', os.environ.get('CP2K_DATA_DIR', '(unset — conda cp2k usually sets this; Stage B will try to autodetect)'))

# Stage A · Generate liquid configs with EMT (free explorer)

32 Cu atoms, FCC 2×2×2 rescaled to liquid density (**8.0 g/cm³ → L ≈ 7.50 Å**).
Melt at 2800 K (superheat kills FCC memory) → linear cool to 1500 K → equilibrate → sample
**300 frames, 50 fs apart** (decorrelated enough for EMT liquid; CP2K stride decimates further).
NVT throughout — density is an *input* here, not a prediction (that's a caveat, see the end).

In [ ]:
%%writefile gen_configs.py
#!/usr/bin/env python
"""Stage A: EMT liquid-Cu config generator. 32 atoms, melt->cool->sample, extxyz out."""
import os, numpy as np
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
from ase.io import write

CU_ROOT   = os.environ.get('CU_ROOT', os.path.expanduser('~/cu-mlip'))
OUT       = os.path.join(CU_ROOT, 'configs.extxyz')
N_FRAMES  = 300
T_MELT    = 2800.0     # K, superheat
T_SAMPLE  = 1500.0     # K, liquid sampling  (same number reused everywhere downstream)
RHO       = 8.00       # g/cm^3 target liquid density
DT_FS     = 2.0
STEPS_MELT, STEPS_COOL, STEPS_EQ = 10000, 10000, 5000
SAMPLE_EVERY = 25      # * 2 fs = 50 fs between frames

# --- 32-atom cell at liquid density ---
atoms = bulk('Cu', 'fcc', a=3.615, cubic=True).repeat((2, 2, 2))   # 32 atoms
m_amu, NA = 63.546, 6.02214076e23
L = (len(atoms) * m_amu / NA / RHO) ** (1/3) * 1e8                 # cm -> Angstrom
atoms.set_cell([L, L, L], scale_atoms=True); atoms.pbc = True
print(f'N={len(atoms)}  L={L:.4f} A  rho={RHO} g/cc')

atoms.calc = EMT()
MaxwellBoltzmannDistribution(atoms, temperature_K=2*T_MELT)
dyn = Langevin(atoms, DT_FS*units.fs, temperature_K=T_MELT, friction=0.02)
print('melting...');  dyn.run(STEPS_MELT)
print('cooling...')
for T in np.linspace(T_MELT, T_SAMPLE, 40):
    dyn.set_temperature(temperature_K=float(T)); dyn.run(STEPS_COOL // 40)
dyn.set_temperature(temperature_K=T_SAMPLE)
print('equilibrating...'); dyn.run(STEPS_EQ)

if os.path.exists(OUT): os.remove(OUT)
n = 0
def snap():
    global n
    write(OUT, atoms, format='extxyz', append=True); n += 1
dyn.attach(snap, interval=SAMPLE_EVERY)
print('sampling...'); dyn.run(N_FRAMES * SAMPLE_EVERY)
print(f'GATE A: wrote {n} frames -> {OUT}')

In [ ]:
!{PY} gen_configs.py

# Stage B · CP2K single-point labels → Δ (resumable, per-frame npz)

Metallic Cu needs: **PBE / GTH-PBE-q11 / DZVP-MOLOPT-SR-GTH**, MGRID CUTOFF 500 Ry,
**Fermi–Dirac SMEAR + ADDED_MOS + Broyden mixing + diagonalization**, MP **2×2×2** k-mesh.
`ENERGY| Total FORCE_EVAL` is the *force-consistent free energy* — sidesteps the
sigma→0-vs-TOTEN issue by construction.

Per frame we store `E_cp2k, F_cp2k, E_emt, F_emt` and the Δs in one npz; already-labeled
frames are skipped, so this cell is safe to interrupt/re-run overnight.

**RAL Step-7 sanity filters, adapted** (integer magnetic moment is meaningless for a
Fermi-Dirac-smeared metal, so it is replaced by a ΔE bound):
1. SCF must report converged within MAX_SCF=200;
2. max |F_cp2k| < 50 eV/Å (garbage-geometry guard);
3. |ΔE|/atom < 10 eV (labeling-blowup guard).
Frames failing any filter get `ok=False` and are excluded from the dataset.

⚠ **Most-likely-to-need-tuning block** (flagged in the handoff): basis/potential file names,
`ADDED_MOS`, `CUTOFF`, and the k-mesh — and note k-point **forces** need a reasonably recent
CP2K; if your build chokes, set `--kpoints 0` for Γ-only as a fallback and say so in the caveats.

In [ ]:
%%writefile label_cp2k.py
#!/usr/bin/env python
"""Stage B: CP2K single-point labeling of EMT configs -> per-frame Delta npz (resumable)."""
import os, sys, re, glob, shutil, argparse, subprocess
import numpy as np
from ase.io import read
from ase.calculators.emt import EMT

HARTREE = 27.211386245988          # eV
AU_F    = HARTREE / 0.529177210903 # eV/A per Hartree/bohr = 51.42208...

CP2K_TEMPLATE = """&GLOBAL
  PROJECT cu_sp
  RUN_TYPE ENERGY_FORCE
  PRINT_LEVEL LOW
&END GLOBAL
&FORCE_EVAL
  METHOD Quickstep
  &DFT
    BASIS_SET_FILE_NAME {basis_file}
    POTENTIAL_FILE_NAME {pot_file}
    &MGRID
      CUTOFF {cutoff}
      REL_CUTOFF 60
    &END MGRID
    &QS
      EPS_DEFAULT 1.0E-10
    &END QS
    &SCF
      SCF_GUESS ATOMIC
      MAX_SCF 200
      EPS_SCF 1.0E-6
      ADDED_MOS {added_mos}
      &SMEAR ON
        METHOD FERMI_DIRAC
        ELECTRONIC_TEMPERATURE [K] {etemp}
      &END SMEAR
      &MIXING
        METHOD BROYDEN_MIXING
        ALPHA 0.4
        NBUFFER 8
      &END MIXING
      &DIAGONALIZATION ON
      &END DIAGONALIZATION
    &END SCF
{kpoint_block}    &XC
      &XC_FUNCTIONAL PBE
      &END XC_FUNCTIONAL
    &END XC
  &END DFT
  &SUBSYS
    &CELL
      A {ax:.10f} {ay:.10f} {az:.10f}
      B {bx:.10f} {by:.10f} {bz:.10f}
      C {cx:.10f} {cy:.10f} {cz:.10f}
      PERIODIC XYZ
    &END CELL
    &COORD
{coords}    &END COORD
    &KIND Cu
      BASIS_SET DZVP-MOLOPT-SR-GTH
      POTENTIAL GTH-PBE-q11
    &END KIND
  &END SUBSYS
  &PRINT
    &FORCES ON
    &END FORCES
  &END PRINT
&END FORCE_EVAL
"""

def find_cp2k():
    for b in ('cp2k.psmp', 'cp2k.ssmp', 'cp2k'):
        p = shutil.which(b)
        if p: return p
    sys.exit('cp2k binary not found (conda install -c conda-forge cp2k)')

def find_data_file(name):
    d = os.environ.get('CP2K_DATA_DIR')
    if d and os.path.exists(os.path.join(d, name)):
        return os.path.join(d, name)
    # conda layout fallback
    for c in glob.glob(os.path.join(sys.prefix, 'share', 'cp2k', 'data', name)):
        return c
    return name   # let CP2K resolve it; will fail loudly if wrong

def parse_output(txt, natoms):
    ok_scf = ('SCF run converged' in txt)
    m = re.search(r'ENERGY\|\s*Total FORCE_EVAL.*?:\s*(-?\d+\.\d+)', txt)
    energy = float(m.group(1)) * HARTREE if m else None
    forces = None
    if 'ATOMIC FORCES in [a.u.]' in txt:
        block = txt.split('ATOMIC FORCES in [a.u.]', 1)[1].splitlines()
        rows = []
        for ln in block:
            if 'SUM OF' in ln: break
            p = ln.split()
            if len(p) >= 6 and p[0].isdigit():
                rows.append([float(p[-3]), float(p[-2]), float(p[-1])])
            if len(rows) == natoms: break
        if len(rows) == natoms:
            forces = np.array(rows) * AU_F
    return ok_scf, energy, forces

def label_frame(atoms, workdir, cp2k_bin, args):
    os.makedirs(workdir, exist_ok=True)
    cell = atoms.cell.array
    coords = ''.join(f'      Cu {p[0]:.10f} {p[1]:.10f} {p[2]:.10f}\n'
                     for p in atoms.get_positions())
    kblock = ''
    if args.kpoints > 0:
        kblock = (f'    &KPOINTS\n      SCHEME MONKHORST-PACK '
                  f'{args.kpoints} {args.kpoints} {args.kpoints}\n    &END KPOINTS\n')
    inp = CP2K_TEMPLATE.format(
        basis_file=find_data_file('BASIS_MOLOPT'), pot_file=find_data_file('GTH_POTENTIALS'),
        cutoff=args.cutoff, added_mos=args.added_mos, etemp=args.etemp, kpoint_block=kblock,
        ax=cell[0,0], ay=cell[0,1], az=cell[0,2],
        bx=cell[1,0], by=cell[1,1], bz=cell[1,2],
        cx=cell[2,0], cy=cell[2,1], cz=cell[2,2], coords=coords)
    with open(os.path.join(workdir, 'cp2k.inp'), 'w') as f: f.write(inp)
    r = subprocess.run([cp2k_bin, '-i', 'cp2k.inp', '-o', 'cp2k.out'],
                       cwd=workdir, capture_output=True, text=True)
    out_path = os.path.join(workdir, 'cp2k.out')
    txt = open(out_path).read() if os.path.exists(out_path) else (r.stdout + r.stderr)
    return parse_output(txt, len(atoms))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--frames',  default=None, help='extxyz (default: CU_ROOT/configs.extxyz)')
    ap.add_argument('--out',     default=None, help='label dir (default: CU_ROOT/labels)')
    ap.add_argument('--nmax',    type=int, default=200, help='frames to label after stride')
    ap.add_argument('--stride',  type=int, default=0,   help='0 = auto from nmax')
    ap.add_argument('--etemp',   type=float, default=1500.0, help='FD electronic T [K] (= ionic T)')
    ap.add_argument('--kpoints', type=int, default=2,   help='MP mesh k x k x k; 0 = Gamma-only')
    ap.add_argument('--cutoff',  type=int, default=500)
    ap.add_argument('--added-mos', type=int, default=80)
    args = ap.parse_args()

    cu_root = os.environ.get('CU_ROOT', os.path.expanduser('~/cu-mlip'))
    frames  = args.frames or os.path.join(cu_root, 'configs.extxyz')
    outdir  = args.out    or os.path.join(cu_root, 'labels')
    os.makedirs(outdir, exist_ok=True)

    traj = read(frames, index=':')
    stride = args.stride or max(1, len(traj) // args.nmax)
    idx = list(range(0, len(traj), stride))[:args.nmax]
    cp2k_bin = find_cp2k()
    print(f'{len(traj)} frames, stride {stride} -> labeling {len(idx)}; cp2k = {cp2k_bin}')

    emt = EMT()
    done = failed = 0
    for j, i in enumerate(idx):
        npz = os.path.join(outdir, f'frame_{i:05d}.npz')
        if os.path.exists(npz):
            try:
                np.load(npz); done += 1; continue     # resumable
            except Exception:
                os.remove(npz)
        atoms = traj[i]
        atoms.calc = emt
        e_emt, f_emt = atoms.get_potential_energy(), atoms.get_forces()
        ok_scf, e_cp2k, f_cp2k = label_frame(atoms, os.path.join(outdir, f'run_{i:05d}'),
                                             cp2k_bin, args)
        ok = bool(ok_scf and e_cp2k is not None and f_cp2k is not None)
        # RAL step-7 filters, adapted for a smeared metal:
        if ok and np.abs(f_cp2k).max() > 50.0:                    ok = False
        if ok and abs(e_cp2k - e_emt) / len(atoms) > 10.0:        ok = False
        if ok:
            np.savez(npz, coord=atoms.get_positions(), box=atoms.cell.array,
                     e_cp2k=e_cp2k, f_cp2k=f_cp2k, e_emt=e_emt, f_emt=f_emt,
                     de=e_cp2k - e_emt, df=f_cp2k - f_emt, ok=True)
            done += 1
        else:
            failed += 1
            print(f'  frame {i}: REJECTED (scf_ok={ok_scf})')
        if (j+1) % 10 == 0:
            print(f'  [{j+1}/{len(idx)}] labeled={done} rejected={failed}')
    print(f'GATE B1: {done} labeled, {failed} rejected -> {outdir}')

if __name__ == '__main__':
    main()

In [ ]:
# Overnight-safe: interrupt any time, re-run to resume. ~150-300 frames is the target.
!{PY} label_cp2k.py --nmax 200

## B2 · Assemble the Δ dataset (DeePMD npy, periodic box, type_map [Cu])
Energies/forces written to the dataset are the **Δs**. 90/10 train/valid split, shuffled with a
fixed seed. The huge absolute CP2K offset is irrelevant — DeePMD fits its own atom-energy bias,
and here it never even sees the absolute energies.

In [ ]:
%%writefile build_delta_dataset.py
#!/usr/bin/env python
"""Stage B2: gather per-frame Delta npz -> DeePMD npy dataset (train/valid)."""
import os, glob, argparse, numpy as np

def dump(frames, outdir):
    os.makedirs(os.path.join(outdir, 'set.000'), exist_ok=True)
    n = len(frames); nat = frames[0]['coord'].shape[0]
    coord  = np.array([f['coord'].reshape(-1) for f in frames])
    box    = np.array([f['box'].reshape(-1)   for f in frames])
    energy = np.array([float(f['de'])         for f in frames])
    force  = np.array([f['df'].reshape(-1)    for f in frames])
    for name, arr in (('coord',coord),('box',box),('energy',energy),('force',force)):
        np.save(os.path.join(outdir, 'set.000', name + '.npy'), arr.astype(np.float64))
    with open(os.path.join(outdir, 'type.raw'), 'w') as f:
        f.write('\n'.join(['0'] * nat) + '\n')
    with open(os.path.join(outdir, 'type_map.raw'), 'w') as f:
        f.write('Cu\n')
    print(f'  {outdir}: {n} frames x {nat} atoms')

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--labels', nargs='+', default=None,
                    help='one or more label dirs (default: CU_ROOT/labels)')
    ap.add_argument('--out', default=None)
    args = ap.parse_args()
    cu_root = os.environ.get('CU_ROOT', os.path.expanduser('~/cu-mlip'))
    label_dirs = args.labels or [os.path.join(cu_root, 'labels')]
    out = args.out or os.path.join(cu_root, 'data_delta')

    frames = []
    for d in label_dirs:
        for p in sorted(glob.glob(os.path.join(d, 'frame_*.npz'))):
            z = np.load(p)
            if bool(z['ok']): frames.append(z)
    print(f'{len(frames)} good frames from {label_dirs}')
    assert len(frames) >= 20, 'too few labeled frames'

    rng = np.random.default_rng(2024)
    order = rng.permutation(len(frames))
    n_val = max(1, len(frames) // 10)
    val   = [frames[i] for i in order[:n_val]]
    trn   = [frames[i] for i in order[n_val:]]

    dump(trn, os.path.join(out, 'train'))
    dump(val, os.path.join(out, 'valid'))

    de = np.array([float(f['de']) for f in frames]) / frames[0]['coord'].shape[0]
    df = np.concatenate([f['df'].reshape(-1) for f in frames])
    print(f'Delta stats: dE/atom mean {de.mean():+.4f} eV, std {de.std():.4f} eV; '
          f'dF rms {np.sqrt((df**2).mean()):.4f} eV/A')
    print('GATE B2 OK ->', out)

if __name__ == '__main__':
    main()

In [ ]:
!{PY} build_delta_dataset.py
# Sanity: the Delta-force RMS printed above is what the DP model must learn. If EMT is a decent
# baseline it should be well below the raw CP2K force scale (~1-3 eV/A in a 1500 K liquid).

# Stage C · Train the Δ-model committee

`input.json` follows the molten-salt paper's recipe scaled to CPU:
- **rcut 6.0 Å** (they chose 7.0 on a bigger box; on our 7.5 Å cell 6.0 already means
  rcut > L/2 — DeePMD handles this correctly via ghost-image replication, *not* minimum image).
- **sel [110]**: ~69 neighbors within 6 Å at 8.0 g/cc, +50% margin (their sizing logic).
- Loss prefactors and **no virial** exactly as in the paper; 100k steps (they show convergence by ~50k).
- Nets scaled down for CPU: embedding [20,40,80], fitting [120,120,120] (paper: [20,50,100]/[240,240,240] —
  fine to restore on GPU).

**Committee of 4** (RAL): identical `input.json` except `seed`; the committee *is* the model —
member 0 drives production, all 4 give the deviation ε.

In [ ]:
%%writefile input.json
{
  "model": {
    "type_map": ["Cu"],
    "descriptor": {"type":"se_e2_a","sel":[110],"rcut_smth":2.0,"rcut":6.0,
                   "neuron":[20,40,80],"axis_neuron":12,"seed":1},
    "fitting_net": {"type":"ener","neuron":[120,120,120],"resnet_dt":true,"seed":1}
  },
  "learning_rate": {"type":"exp","start_lr":0.001,"stop_lr":3.5e-08,"decay_steps":5000},
  "loss": {"type":"ener","start_pref_e":0.02,"limit_pref_e":1,
           "start_pref_f":1000,"limit_pref_f":1,"start_pref_v":0,"limit_pref_v":0},
  "training": {
    "training_data":   {"systems":["DATA_TRAIN"],"batch_size":"auto"},
    "validation_data": {"systems":["DATA_VALID"],"batch_size":"auto","numb_btch":1},
    "numb_steps":100000,"seed":1,"disp_file":"lcurve.out","disp_freq":500,"save_freq":5000
  }
}

In [ ]:
# Train the committee. Reduce N_COMMITTEE to 2 if CPU time hurts (keep >=2: no committee, no sigma).
import json, os, subprocess, glob
N_COMMITTEE = 4
CU_ROOT  = os.environ['CU_ROOT']
MODELS   = os.path.join(CU_ROOT, 'models')
dp_bin   = os.path.join(os.path.dirname(PY), 'dp')
os.environ['MPLBACKEND'] = 'Agg'

base = json.load(open('input.json'))
base['training']['training_data']['systems']   = [os.path.join(CU_ROOT, 'data_delta', 'train')]
base['training']['validation_data']['systems'] = [os.path.join(CU_ROOT, 'data_delta', 'valid')]

for i in range(N_COMMITTEE):
    d = os.path.join(MODELS, f'c{i}'); os.makedirs(d, exist_ok=True)
    if glob.glob(os.path.join(d, 'graph.pb')):
        print(f'c{i}: graph.pb exists, skipping'); continue
    cfg = json.loads(json.dumps(base))
    for k in (('model','descriptor'), ('model','fitting_net'), ('training',)):
        node = cfg
        for kk in k: node = node[kk]
        node['seed'] = 1000 + i
    json.dump(cfg, open(os.path.join(d, 'input.json'), 'w'), indent=2)
    print(f'--- training committee member c{i} ---')
    subprocess.run([dp_bin, 'train', 'input.json'], cwd=d, check=True)
    subprocess.run([dp_bin, 'freeze', '-o', 'graph.pb'], cwd=d, check=True)

pbs = sorted(glob.glob(os.path.join(MODELS, 'c*', 'graph.pb')))
print('GATE C', 'OK' if len(pbs) >= 2 else 'FAILED', '-', pbs)

## C2 · Validate against the paper's RMSE bar
`dp test` on the held-out valid set. Targets (Xu et al.): **energy < ~1.3 meV/atom**,
**force < ~4.5×10⁻² eV/Å** — remember these RMSEs are on the **Δ**, i.e. exactly the error of the
*total* EMT+DP model vs CP2K, since EMT is deterministic.

In [ ]:
import subprocess, os, glob
dp_bin = os.path.join(os.path.dirname(PY), 'dp')
valid  = os.path.join(os.environ['CU_ROOT'], 'data_delta', 'valid')
for pb in sorted(glob.glob(os.path.join(os.environ['CU_ROOT'], 'models', 'c*', 'graph.pb'))):
    print('===', pb)
    r = subprocess.run([dp_bin, 'test', '-m', pb, '-s', valid, '-n', '9999'],
                       capture_output=True, text=True)
    print('\n'.join(l for l in (r.stdout + r.stderr).splitlines()
                    if 'RMSE' in l or 'rmse' in l))

# Stage D · Production MD on the replicated box (256 atoms)

Train small, **run big**: the labeled 32-atom liquid is replicated 2×2×2 → 256 atoms
(replication imprints artificial periodicity; the 10 ps equilibration washes it out).
Calculator = `SumCalculator([EMT, DP(c0)])`. Langevin NVT at **1500 K** (same number as
sampling, labeling e-temp — temperature-consistency discipline).

**RAL deviation monitor:** every 200 steps, ε = max over atoms of the committee force std
(Achar Eq. 1; EMT cancels, so ε is computed on the Δ-models directly). Frames with
0.1 < ε < 1.0 eV/Å are appended to `candidates.extxyz` — the hook for an AL round.

In [ ]:
%%writefile production.py
#!/usr/bin/env python
"""Stage D: EMT+DP(Delta) production NVT MD on 2x2x2 replicated box + committee monitor."""
import os, glob, numpy as np
from ase.io import read, write
from ase.io.trajectory import Trajectory
from ase.calculators.emt import EMT
from ase.calculators.mixing import SumCalculator
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
from deepmd.calculator import DP

CU_ROOT   = os.environ.get('CU_ROOT', os.path.expanduser('~/cu-mlip'))
T         = 1500.0
DT_FS     = 2.0
STEPS_EQ  = 5000      # 10 ps
STEPS_PROD= 20000     # 40 ps
TRAJ_EVERY, DEV_EVERY = 10, 200
EPS_LO, EPS_HI = 0.1, 1.0   # RAL candidate window (eV/A)

pbs = sorted(glob.glob(os.path.join(CU_ROOT, 'models', 'c*', 'graph.pb')))
assert pbs, 'no committee models found'
print('committee:', pbs)

atoms = read(os.path.join(CU_ROOT, 'configs.extxyz'), index=-1).repeat((2, 2, 2))
print(f'production box: {len(atoms)} atoms, L = {atoms.cell.lengths()[0]:.3f} A')
atoms.calc = SumCalculator([EMT(), DP(model=pbs[0])])

dp_committee = [DP(model=p) for p in pbs]
def committee_eps(a):
    F = []
    for calc in dp_committee:
        b = a.copy(); b.calc = calc
        F.append(b.get_forces())
    F = np.array(F)                             # (m, N, 3)
    var = ((F - F.mean(0))**2).mean(0).sum(-1)  # per-atom <|dF|^2>
    return float(np.sqrt(var).max())            # Achar et al. eq 1

MaxwellBoltzmannDistribution(atoms, temperature_K=T)
dyn = Langevin(atoms, DT_FS*units.fs, temperature_K=T, friction=0.02)
print('equilibrating 10 ps...'); dyn.run(STEPS_EQ)

traj = Trajectory(os.path.join(CU_ROOT, 'production.traj'), 'w', atoms)
cand = os.path.join(CU_ROOT, 'candidates.extxyz')
if os.path.exists(cand): os.remove(cand)
devlog = open(os.path.join(CU_ROOT, 'deviation.log'), 'w')
devlog.write('# step  eps_max(eV/A)  T(K)\n')

state = {'step': 0, 'ncand': 0}
def on_traj(): traj.write()
def on_dev():
    eps = committee_eps(atoms)
    Tk  = atoms.get_temperature()
    devlog.write(f'{state["step"]:8d} {eps:10.4f} {Tk:9.1f}\n'); devlog.flush()
    if EPS_LO < eps < EPS_HI:
        write(cand, atoms, format='extxyz', append=True); state['ncand'] += 1
def tick(): state['step'] += DEV_EVERY  # attached at DEV_EVERY

dyn.attach(on_traj, interval=TRAJ_EVERY)
dyn.attach(on_dev,  interval=DEV_EVERY)
dyn.attach(tick,    interval=DEV_EVERY)
print('production 40 ps...'); dyn.run(STEPS_PROD)
traj.close(); devlog.close()
print(f'GATE D OK: production.traj written; {state["ncand"]} candidate frames -> candidates.extxyz')

In [ ]:
!{PY} production.py

## D2 · Analysis — g(r) and self-diffusion vs reference
- **g(r)**: MIC pair histogram averaged over production frames. Liquid Cu near 1500 K:
  first peak at **r₁ ≈ 2.5 Å** (X-ray, Waseda-type data ~2.50–2.57 Å), first minimum ~3.4–3.6 Å,
  first-shell coordination **N₁ ≈ 11–12** from ∫g(r) (same n(r) integral as the salt paper's Table 1).
- **D**: Einstein slope of the multi-origin MSD, mid-window fit. Liquid Cu:
  **D ≈ 4×10⁻⁵ cm²/s** near melting (1358 K), ~5–7×10⁻⁵ at 1500 K. Order of magnitude and
  temperature-trend agreement is the success criterion — exact match is not expected from
  PBE + this data budget, and NVT at an assumed density biases both observables slightly.

In [ ]:
%%writefile analyze.py
#!/usr/bin/env python
"""Stage D2: g(r) + MSD->D from production.traj, with reference markers."""
import os, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from ase.io.trajectory import Trajectory

CU_ROOT = os.environ.get('CU_ROOT', os.path.expanduser('~/cu-mlip'))
DT_PS   = 2.0e-3 * 10          # traj written every 10 steps of 2 fs -> 0.02 ps/frame

traj = Trajectory(os.path.join(CU_ROOT, 'production.traj'))
print(len(traj), 'frames')

# ---------- g(r) ----------
L   = traj[0].cell.lengths()[0]
N   = len(traj[0])
rmax, nbins = L/2, 200
edges = np.linspace(0, rmax, nbins+1); rc = 0.5*(edges[1:]+edges[:-1])
hist = np.zeros(nbins); nf = 0
for a in traj[::10]:
    pos = a.get_positions(); cell = a.cell.array
    d = pos[:, None, :] - pos[None, :, :]
    d -= np.round(d @ np.linalg.inv(cell)) @ cell        # MIC
    r = np.sqrt((d**2).sum(-1))[np.triu_indices(N, 1)]
    hist += np.histogram(r, bins=edges)[0]; nf += 1
rho = N / traj[0].get_volume()
shell = 4/3*np.pi*(edges[1:]**3 - edges[:-1]**3)
g = hist / (nf * 0.5 * N * rho * shell)     # upper-triangle pairs -> 0.5*N normalization
nr = np.cumsum(rho * g * shell)             # coordination integral n(r)
i1 = np.argmax(g); imin = i1 + np.argmin(g[i1:np.searchsorted(rc, 4.5)])
print(f'g(r): first peak r1 = {rc[i1]:.3f} A (ref ~2.50-2.57), '
      f'first min = {rc[imin]:.3f} A, N1 = {nr[imin]:.2f} (ref ~11-12)')

# ---------- MSD -> D ----------
pos = np.array([a.get_positions() for a in traj])   # ASE MD does not wrap -> continuous
nfr = len(pos); max_lag = nfr // 2
origins = range(0, nfr - max_lag, 5)
msd = np.zeros(max_lag)
for lag in range(1, max_lag):
    s = 0.0
    for t0 in origins:
        dr = pos[t0+lag] - pos[t0]
        s += (dr**2).sum(-1).mean()
    msd[lag] = s / len(list(origins))
t = np.arange(max_lag) * DT_PS
lo, hi = max_lag//4, max_lag                       # mid-to-late window fit
slope = np.polyfit(t[lo:hi], msd[lo:hi], 1)[0]     # A^2/ps
D = slope / 6 * 1e-4                               # -> cm^2/s
print(f'D = {D:.3e} cm^2/s  (liquid Cu ref: ~4e-5 near Tm, ~5-7e-5 at 1500 K)')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rc, g); ax[0].axvline(2.55, ls='--', c='gray', label='ref r1~2.55 A')
ax[0].set(xlabel='r (A)', ylabel='g(r)', title='liquid Cu g(r), 1500 K'); ax[0].legend()
ax[1].plot(t, msd); ax[1].plot(t[lo:hi], np.polyval(np.polyfit(t[lo:hi], msd[lo:hi], 1), t[lo:hi]), 'r--')
ax[1].set(xlabel='t (ps)', ylabel='MSD (A$^2$)', title=f'D = {D:.2e} cm$^2$/s')
out = os.path.join(CU_ROOT, 'analysis.png')
fig.tight_layout(); fig.savefig(out, dpi=140)
print('GATE D2 OK ->', out)

In [ ]:
!{PY} analyze.py
from IPython.display import Image
import os
Image(os.path.join(os.environ['CU_ROOT'], 'analysis.png'))

# Stage E · (Optional) AL round from production candidates

The deviation monitor already wrote `candidates.extxyz` (0.1 < ε < 1.0 eV/Å window, RAL
classification). One AL round = relabel candidates → merge → retrain — the DP-GEN/RAL loop
your `al.py` implements, but with the *explorer being production MD itself*:

```bash
{PY} label_cp2k.py --frames $CU_ROOT/candidates.extxyz --out $CU_ROOT/labels_al --stride 1 --nmax 50
{PY} build_delta_dataset.py --labels $CU_ROOT/labels /home/$USER/cu-mlip/labels_al
# delete models/c*/graph.pb (or move CU_ROOT/models aside) and re-run the Stage C training cell
```
If ε stays < 0.1 eV/Å for essentially the whole production run (check `deviation.log`), the
committee agrees everywhere it goes and an AL round buys nothing — that is the *good* outcome
for an equilibrium-liquid validation. The RAL thresholds were tuned for reactive PES exploration;
for a Δ-model on a mild liquid you may find even 0.1 conservative.

In [ ]:
# Quick look at the deviation history
import numpy as np, matplotlib.pyplot as plt, os
f = os.path.join(os.environ['CU_ROOT'], 'deviation.log')
d = np.loadtxt(f)
if d.size:
    d = np.atleast_2d(d)
    plt.figure(figsize=(8,3)); plt.plot(d[:,0]*0.002, d[:,1])
    plt.axhline(0.1, ls='--', c='g'); plt.axhline(1.0, ls='--', c='r')
    plt.xlabel('t (ps)'); plt.ylabel(r'$\epsilon$ max force dev (eV/$\AA$)')
    plt.title(f'committee deviation — {(d[:,1]>0.1).mean()*100:.1f}% frames above 0.1 eV/A')
    plt.tight_layout(); plt.show()

## Reading the result & caveats

**Success criteria (in order):**
1. `dp test` RMSE at or near the paper bar (<~1.3 meV/atom, <45 meV/Å) — on the Δ, which *is*
   the total-model error vs CP2K.
2. g(r): r₁ ≈ 2.5 Å, N₁ ≈ 11–12, liquid-like (no crystalline split peaks).
3. D within ~2× of 4–7×10⁻⁵ cm²/s at 1500 K.
4. Committee ε mostly < 0.1 eV/Å through production.

**Caveats — do not paper over these:**
- **NVT at an assumed density.** Density here is an input (8.0 g/cc), not a prediction; the salt
  paper predicted density via NPT, which needs the **virial** — which we deliberately did not train
  (their own ablation, and Δ-virial from EMT+DP is untested). g(r)/D carry a small density bias.
- **CP2K block is the tuning hotspot**: basis/potential names must match your build's data files;
  `ADDED_MOS 80` and `CUTOFF 500` are sane starting points for q11 Cu, not gospel; k-point forces
  require a recent CP2K — Γ-only (`--kpoints 0`) is the fallback, at the price of worse metallic
  sampling in a 7.5 Å cell.
- **Electronic temperature** is set = ionic (1500 K) for a consistent free-energy surface; some
  practitioners fix it lower. Whatever you pick, keep it *constant across all labels*.
- **EMT is a fitted toy for Cu.** Good baseline, but the Δ inherits none of its guarantees far
  from fcc/liquid geometries — the committee monitor is the guardrail.
- **rcut (6.0) > L/2 (3.75)** on the training cell: fine in DeePMD (full periodic images), but it
  means descriptor environments in training partially self-correlate; the 256-atom production box
  does not have this issue, which is exactly why validation happens there.
- **CP2K reference only.** Do not merge VASP-labeled frames into `data_delta` later.
- This is a **methodology validation**, not production Cu — the deliverable is confidence in the
  labeling→Δ-training→committee→production→observables chain before Cu–In.